# Launching ND AI Lab

The same thing `launch_nd_ai_lab.py` does, from a notebook.

Set `DATASET` below to choose the project. Each one is used **in place**,
so anything you annotate is written back into it.

## Qt event loop

napari needs a Qt event loop. In a notebook that comes from `%gui qt`,
not from `napari.run()` -- calling `run()` here would block the kernel.

In [13]:
%gui qt

## Choose the project

| dataset | what it is |
|---|---|
| `bees` | seven comb images, already labelled |
| `ladybugs` | eleven photographs, no annotations yet |
| `pollen_count` | ten slides, already labelled |

`ladybugs` starts empty, so ND AI Lab creates the annotation folders on
first save. The other two open with their existing labels.

In [7]:
DATASET = 'bees'   # 'bees' | 'ladybugs' | 'pollen_count'

from pathlib import Path

# Works whether the kernel starts in notebooks/ or at the repo root.
for candidate in (Path.cwd() / "data", Path.cwd() / "notebooks" / "data"):
    if (candidate / DATASET).is_dir():
        project = candidate / DATASET
        break
else:
    raise FileNotFoundError(f"{DATASET} not found from {Path.cwd()}")

images = sorted(project.glob('*.png')) + sorted(project.glob('*.jpg'))
print(project)
print(len(images), 'images')

C:\Users\bnort\work\ImageJ2022\tnia\i2k-2026\notebooks\data\bees
7 images


## Launch

`register_all=True` registers every segmenter and augmenter, so the
dropdowns are populated. Pass `False` to register only what you import
yourself.

In [10]:
import napari
from napari_ai_lab.apps.nd_ai_lab_launcher import launch_nd_ai_lab

from napari_ai_lab.apps.profiles import list_profiles, get_profile

list_profiles()

['2d-instance-skop', 'all']

In [12]:
viewer = napari.Viewer()

ai_lab, sequence_viewer, model = launch_nd_ai_lab(
    viewer,
    project,
    viewer_type="sequence",
    axes_to_collapse="C",
    axis_types="NYXC",
    register_all=True,
    profile="all",
)

print('ND AI Lab launched on', project.name)

Registering profile: all
Registered global segmenter: StarDist2D (scikit-ops)
Registered global segmenter: Cellpose3 (scikit-ops)
Registered global segmenter: Cellpose4 (scikit-ops)
Registered global segmenter: CellCastStardistSegmenter
Registered global segmenter: ThresholdSegmenter
Registered global segmenter: MicroSamSegmenter
Registered global segmenter: MonaiUNetSegmenter
Registered global segmenter: MonaiUNetSegmenter3D
Registered global segmenter: MicrosamYoloSegmenter
Registered global segmenter: SkImageWatershedSegmenter
Registered interactive segmenter: Otsu2D
Registered interactive segmenter: Otsu3D
Registered interactive segmenter: SAM3D
Registered interactive segmenter: SAMSphere3D
Registered interactive segmenter: RegionGrow3D
Registered interactive segmenter: FeatureRegionGrow3D
Registered interactive segmenter: AnisotropicSphereFit3D
Registered interactive segmenter: HoughSphereFit3D
Registered augmenter: SimpleAugmenter
Registered augmenter: AlbumentationsAugmenter
Con

C:\Users\bnort\work\ImageJ2022\tnia\napari-ai-lab\pixi\pytorch_napari\.pixi\envs\default\Lib\site-packages\napari\layers\utils\style_encoding.py:251: RuntimeWarning: Applying the encoding failed. Using the safe fallback value instead.
  warnings.warn(


Viewer closing detected via closeEvent


## Why those arguments

| argument | why |
|---|---|
| `viewer_type="sequence"` | all three sets hold images of different shapes, so they cannot be one stacked array |
| `axes_to_collapse="C"` | colour is not an axis to annotate along; labels are 2D per image |
| `axis_types="NYXC"` | N images, each Y by X with a colour axis |

The same three values suit all three datasets. A project of equal-sized
images could use `viewer_type="stacked"` instead.

## Using it

The **Sequence Viewer** at the bottom moves between images. The **AI Lab**
dock on the right has Label, Augment and Segment.

Interactive segmentation lives on the Label tab. SAM3D there needs
micro_sam, which the pixi environment has and the pip fallback does not.